# Beyond Class Imbalance: Models, Thresholds and Better Decisions

**Optional self-learning | Supervised Machine Learning | ~45–60 minutes**

**Guiding question:** Does an imbalanced dataset require SMOTE, and does a better model automatically lead to better business decisions?

This notebook is **fully worked**: run the cells, inspect the figures, and reflect on the interpretations. No exercises or TODOs.

**Learning goals**
- Question whether resampling is necessary.
- Compare a linear baseline (Logistic Regression) with a flexible gradient-boosted tree model (XGBoost).
- See how **SMOTENC** handles mixed numerical and categorical predictors.
- Separate model discrimination, threshold selection, and business costs.

**Prerequisite:** the main notebook *7.5 — Class Imbalance: Metrics, Resampling and Better Decisions*.

## 0. Setup and data

Install the extra packages in your notebook environment if necessary:

```bash
conda install -c conda-forge xgboost imbalanced-learn
```

We reuse the **[Online Shoppers Purchasing Intention Dataset](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset)** (12,330 sessions; purchase is the positive class). Source: Sakar et al. (2018), *Real-time prediction of online shoppers' purchasing intention using multilayer perceptron and LSTM recurrent neural networks*. The UCI dataset is licensed **CC BY 4.0**; attribution is required. Data are fetched from UCI at runtime, not bundled in this notebook.

**Methodological rule:** split off the test set first; perform all preprocessing and resampling **inside the training folds**. The test set remains untouched until the final evaluation.

In [ ]:
import io
import zipfile
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix, roc_curve, classification_report,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder
from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_URL = 'https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip'

with urlopen(DATA_URL, timeout=45) as response:
    raw = response.read()
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    names = [name for name in archive.namelist() if name.lower().endswith('.csv')]
    if len(names) != 1:
        raise ValueError(f'Expected one CSV, found {names}')
    with archive.open(names[0]) as handle:
        data = pd.read_csv(handle)

numeric_cols = [
    'Administrative', 'Administrative_Duration', 'Informational',
    'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
]
categorical_cols = [
    'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType',
    'VisitorType', 'Weekend',
]
X = data.drop(columns='Revenue')
y = data['Revenue'].astype(bool)
assert set(numeric_cols + categorical_cols) == set(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.20, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
print('Training shape:', X_train.shape, '| purchase prevalence:', f'{y_train.mean():.1%}')
print('Test shape:', X_test.shape, '| purchase prevalence:', f'{y_test.mean():.1%}')

## 1. Do we really need SMOTE? (~10 min)

**No.** Class imbalance is a property of the data, not a command to generate synthetic observations. Oversampling can help some learners in some settings, but it can also change the learned decision boundary, increase training time, or fail to improve the metric we care about.

A useful research prompt is **[To SMOTE, or not to SMOTE?](https://arxiv.org/search/?query=To+SMOTE%2C+or+not+to+SMOTE&searchtype=all)**. Read the study critically: its findings are evidence from particular experimental conditions, **not** a universal theorem that SMOTE is always harmful or always beneficial.

We will compare **three options** for each model: no intervention, class weighting, and SMOTENC. We will not assume that a perfectly balanced training set is desirable.

**Metrics:** Average Precision (AP) summarizes the precision–recall curve; ROC-AUC summarizes ranking performance across thresholds. Neither requires choosing a decision threshold. AP is **not identical** to trapezoidal PR-AUC.

**Why SMOTENC rather than ordinary SMOTE?** The data mix numeric features and categorical features, including integer-coded categories. Interpolating one-hot dummy variables with ordinary SMOTE can create invalid fractional categories. [SMOTENC documentation](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html).

## 2. Linear model vs gradient boosting (~15–20 min)

We use **Logistic Regression** as a relatively simple linear reference and **XGBoost** as a more flexible tree-boosting model. “Weak” and “strong” here describe model flexibility informally: Logistic Regression is not intrinsically a poor classifier, and XGBoost is not guaranteed to win.

For a fair comparison, use the **same stratified folds** for every strategy. Resampling happens **only within each training fold**.

**XGBoost documentation:** [XGBClassifier Python API](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier) and [XGBoost parameter reference](https://xgboost.readthedocs.io/en/stable/parameter.html). In this experiment, pay particular attention to `n_estimators`, `max_depth`, `learning_rate`, and `scale_pos_weight`. The last parameter increases the weight assigned to positive examples; a common starting point for binary classification is `number of negative examples / number of positive examples`, but it is not necessarily optimal.

### How the SMOTENC pipeline works

1. `MinMaxScaler` scales numeric features; `OrdinalEncoder` gives categorical features temporary integer codes (including a code for unseen categories).
2. `SMOTENC` receives **both** numeric and categorical features and knows which positions are categorical. It creates synthetic samples without linearly interpolating categorical codes as if they were continuous.
3. `OneHotEncoder` encodes the categorical columns **after** resampling, before the classifier.

**Why `sparse_threshold=0` in the `ColumnTransformer` steps?** One-hot encoding often creates a sparse matrix (mostly zeros). Setting `sparse_threshold=0` makes `ColumnTransformer` return a **dense** array, simplifying compatibility with subsequent steps. The default is `0.3`: if any transformer produces sparse output, the combined result is sparse when its overall density is below that threshold. Dense arrays can consume substantially more memory when there are many categories, so this is a convenient choice for this small example, not a general recommendation.

`SMOTENC` handles the mixed feature types jointly; it does **not** independently apply SMOTE to each column. The intermediate ordinal codes are a representation for SMOTENC, not a claim that category codes are ordered quantities.

For illustration we use `sampling_strategy=0.5`: the final minority/majority **count ratio** is 0.5 (roughly 33.3% minority prevalence), **not** 50% prevalence. We choose this ratio to illustrate partial resampling, not because it is optimal.

In [ ]:
# Column order is important: numeric columns first, categorical columns second.
# These are positional indices in the output of the first ColumnTransformer.
cat_indices = list(range(len(numeric_cols), len(numeric_cols) + len(categorical_cols)))
num_indices = list(range(len(numeric_cols)))

before_sampler = ColumnTransformer([
    ('num', MinMaxScaler(), numeric_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols),
], sparse_threshold=0)

after_sampler = ColumnTransformer([
    ('num', 'passthrough', num_indices),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_indices),
], sparse_threshold=0)

# SMOTENC is intentionally placed BETWEEN ordinal encoding and one-hot encoding.
def make_pipeline(model, sampler=None):
    steps = [('before_sampler', before_sampler)]
    if sampler is not None:
        steps.append(('sampler', sampler))
    steps += [('after_sampler', after_sampler), ('model', model)]
    return ImbPipeline(steps)


def logistic(weighted=False):
    return LogisticRegression(max_iter=1200, class_weight='balanced' if weighted else None)


def xgboost(weighted=False):
    # The positive/negative class count ratio is computed from TRAINING labels.
    # For exact fold-specific weights, one could fit a custom wrapper inside each fold.
    # Here it is a fixed, training-set-derived illustration.
    pos_weight = (y_train.eq(False).sum() / y_train.eq(True).sum()) if weighted else 1.0
    return XGBClassifier(
        n_estimators=120, max_depth=3, learning_rate=0.07,
        subsample=0.9, colsample_bytree=0.9,
        tree_method='hist', eval_metric='logloss',
        scale_pos_weight=pos_weight, random_state=RANDOM_STATE,
        n_jobs=2,
    )


def smotenc():
    return SMOTENC(
        categorical_features=cat_indices,
        sampling_strategy=0.5,
        random_state=RANDOM_STATE,
        k_neighbors=5,
    )

models = {
    'LR | baseline': make_pipeline(logistic()),
    'LR | class weights': make_pipeline(logistic(weighted=True)),
    'LR | SMOTENC': make_pipeline(logistic(), smotenc()),
    'XGB | baseline': make_pipeline(xgboost()),
    'XGB | class weights': make_pipeline(xgboost(weighted=True)),
    'XGB | SMOTENC': make_pipeline(xgboost(), smotenc()),
}

rows = []
for name, estimator in models.items():
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv,
        scoring={'ap': 'average_precision', 'roc_auc': 'roc_auc'},
        n_jobs=1, error_score='raise',
    )
    rows.append({
        'strategy': name,
        'AP mean': scores['test_ap'].mean(),
        'AP std': scores['test_ap'].std(),
        'ROC-AUC mean': scores['test_roc_auc'].mean(),
    })
results = pd.DataFrame(rows).sort_values('AP mean', ascending=False).reset_index(drop=True)
display(results.round(3))

**Interpret the table rather than assuming a winner:**

- Did the more flexible model outperform Logistic Regression **on this dataset**?
- Did SMOTENC improve AP relative to the same model without resampling?
- Did weighting help both models in the same way?
- Are the AP differences large relative to the fold-to-fold variability?

**Cautions:** Three folds provide a compact illustration, not a statistically definitive model comparison. `class_weight='balanced'` in Logistic Regression and `scale_pos_weight` in XGBoost are different mechanisms; XGBoost's weight here is computed once from the outer training set rather than recalculated separately per CV fold. Class weighting and SMOTENC can also change probability calibration.

**Key idea:** Choose a strategy based on validation evidence and the task, not on the existence of imbalance alone.

## 3. Choosing better thresholds (~15 min)

A classifier's ranking ability and its **decision rule** are different questions. A threshold of 0.5 is not automatically the best business decision.

**Youden's J** is

\[
J(t) = \mathrm{TPR}(t) - \mathrm{FPR}(t)
= \mathrm{Sensitivity}(t) + \mathrm{Specificity}(t) - 1.
\]

Maximizing J balances sensitivity and specificity in a particular sense; it does **not** incorporate business costs or the operational prevalence directly.

Suppose a false positive (unnecessary marketing contact) costs **1 unit**, while a false negative (missing a potential buyer) costs **5 units**. This is a **fictional illustrative scenario**, not a claim about actual marketing economics.

\[
C(t) = 1\cdot FP(t) + 5\cdot FN(t)
\]

We choose thresholds using **out-of-fold predictions from training data only**, then reserve the test set for a single final evaluation. We use the best AP configuration as a demonstration; in a production workflow, model and threshold selection would typically be nested or supported by a separate calibration/validation split.

In [ ]:
best_name = results.loc[0, 'strategy']
best_model = models[best_name]
print('Illustrative model selected by CV AP:', best_name)

# Every training observation is predicted by a model that did not train on it.
oof_matrix = cross_val_predict(best_model, X_train, y_train, cv=cv, method='predict_proba', n_jobs=1)
positive_col = list(np.unique(y_train)).index(True)
oof_scores = oof_matrix[:, positive_col]
y_train_array = y_train.to_numpy(dtype=bool)

fpr, tpr, roc_thresholds = roc_curve(y_train_array, oof_scores, pos_label=True)
finite = np.isfinite(roc_thresholds)
youden_values = tpr - fpr
youden_index = np.argmax(np.where(finite, youden_values, -np.inf))
youden_threshold = float(roc_thresholds[youden_index])

# Candidate thresholds include all distinct observed scores and the endpoints.
threshold_grid = np.unique(np.r_[0.0, oof_scores, 1.0])

def counts_and_cost(y_true, scores, threshold, fp_cost=1, fn_cost=5):
    pred = scores >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[False, True]).ravel()
    return {'threshold': threshold, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'cost': fp_cost * fp + fn_cost * fn}

cost_curve = pd.DataFrame([
    counts_and_cost(y_train_array, oof_scores, t) for t in threshold_grid
])
cost_threshold = float(cost_curve.loc[cost_curve['cost'].idxmin(), 'threshold'])

selected_thresholds = {'Default 0.50': 0.5, 'Youden J': youden_threshold,
                       'Minimum illustrative cost': cost_threshold}
summary = pd.DataFrame([
    {'rule': name, **counts_and_cost(y_train_array, oof_scores, t)}
    for name, t in selected_thresholds.items()
])
display(summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr[finite], youden_values[finite], marker='.', linewidth=1)
axes[0].axvline(fpr[youden_index], linestyle='--', label=f'Max J = {youden_values[youden_index]:.2f}')
axes[0].set(xlabel='False positive rate', ylabel='Youden J', title='Youden J across ROC points')
axes[0].legend()
axes[1].plot(cost_curve['threshold'], cost_curve['cost'])
for label, t in selected_thresholds.items():
    axes[1].axvline(t, linestyle='--', label=f'{label}: {t:.2f}')
axes[1].set(xlabel='Decision threshold', ylabel='Total illustrative cost', title='Cost across thresholds')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Important:** These costs are counts multiplied by assumed unit costs, **not** calibrated expected monetary losses. A model can rank cases well (high AP/ROC-AUC) while its probability estimates are poorly calibrated. If a decision relies on literal probabilities or expected value, [probability calibration](https://scikit-learn.org/stable/modules/calibration.html) deserves a separate validation step.

The curves are derived from training-set out-of-fold predictions, not from the held-out test set. We did not resample any validation fold.

## 4. Putting it all together: final worked example (~10 min)

We now fit the selected pipeline on the **full training set**, then evaluate it **once** on the untouched test set. We compare the three thresholds selected earlier. This is an illustrative workflow, not a guarantee that the selected strategy or thresholds will generalize optimally.

In real applications, costs should come from domain experts, and the operational decision may depend on contact capacity, budget, fairness constraints, and calibration.

In [ ]:
best_model.fit(X_train, y_train)
test_matrix = best_model.predict_proba(X_test)
# The positive class is True. sklearn-compatible classifiers order classes as [False, True].
test_scores = test_matrix[:, list(best_model.named_steps['model'].classes_).index(True)]

print('Test AP:', round(average_precision_score(y_test, test_scores), 3))
print('Test ROC-AUC:', round(roc_auc_score(y_test, test_scores), 3))

final_rows = []
for name, threshold in selected_thresholds.items():
    c = counts_and_cost(y_test.to_numpy(dtype=bool), test_scores, threshold)
    pred = test_scores >= threshold
    final_rows.append({
        'rule': name, 'threshold': threshold,
        'precision': precision_score(y_test, pred, pos_label=True, zero_division=0),
        'recall': recall_score(y_test, pred, pos_label=True, zero_division=0),
        'FP': c['FP'], 'FN': c['FN'], 'illustrative cost': c['cost'],
    })
display(pd.DataFrame(final_rows).round(3))

## Final takeaways

1. **Imbalance does not automatically require SMOTE.** Test the effect on the metric that matters.
2. **Different models respond differently.** A flexible boosted-tree classifier may or may not benefit from resampling or weighting.
3. **SMOTENC is a useful option for mixed data**, but synthetic examples still require scrutiny.
4. **Model selection and threshold selection are separate decisions.** Choose both without using the test set to tune them.
5. **A good score is not the same as a good decision.** Business costs and calibration can change what “best” means.

**Further reading:** [SMOTENC API](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html) · [XGBoost Python API](https://xgboost.readthedocs.io/en/stable/python/python_api.html) · [Scikit-learn: tuning the decision threshold](https://scikit-learn.org/stable/modules/classification_threshold.html) · [Probability calibration](https://scikit-learn.org/stable/modules/calibration.html).